# 🗜️ Memory Optimizations: Keeping Conversations Affordable

The previous notebook gave our agent perfect recall by replaying the entire conversation on
every turn. That works — right up until it doesn't. Every extra token costs money and adds
latency, and both grow **linearly with the length of the conversation**.

This notebook builds three memory strategies on the same customer-support scenario and
compares what each one costs.

## Learning Objectives
In this notebook, you will learn:
1. **Sequential memory** - the full-history baseline, and exactly how it degrades
2. **Sliding window memory** - capping context with `RemoveMessage`, and what you lose
3. **Summarization memory** - compressing old turns instead of discarding them
4. **Measuring the trade-off** - instrumenting latency and token usage per turn

## Prerequisites
- **`01_Memory_and_Conversational_Agent.ipynb`** — checkpointers and threads
- A `.env` file at the repo root with credentials for your configured provider
- Packages: `langgraph`, `langgraph-checkpoint-sqlite`

### Key Concepts:
- **Reducer**: the function that merges a node's return value into state (`add` vs `add_messages`)
- **`RemoveMessage`**: a sentinel that tells `add_messages` to *delete* a message by id
- **Context window**: the token budget one model call must fit inside

> **The trade-off in one line**: sequential never forgets but never stops growing; sliding
> window is cheap but genuinely forgets; summarization keeps the gist at the cost of an
> extra LLM call per turn.

---
## 🔧 Part 0: Setup

Imports, credentials, and the shared LLM. Everything downstream reuses this one client.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Credentials
# ============================================================================

# --- Standard library ---
import os
import sys
import time
from operator import add
from typing import Annotated

# --- Third-party ---
from dotenv import load_dotenv
from typing_extensions import TypedDict

# --- LangChain ---
from langchain_core.messages import (
    AnyMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
)

# --- LangGraph ---
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

# --- Project helpers ---
sys.path.append(os.path.abspath("../../../.."))
from helpers import get_llm

load_dotenv()

print("✅ Imports and environment loaded successfully!")

In [ ]:
# ============================================================================
# LLM INITIALIZATION
# ============================================================================
# Platform-aware defaults: Databricks on macOS, Groq on Windows.

llm = get_llm()

print(f"🤖 LLM initialized: {getattr(llm, 'model_name', type(llm).__name__)}")

In [ ]:
# ============================================================================
# INSTRUMENTATION: Measure Every Turn
# ============================================================================
# Each strategy is judged on the same two numbers, so we time the call and read
# token usage off the reply in one place.

def timed_invoke(graph, state: dict, config: dict, channel: str = "messages"):
    """Invoke a graph, print the exchange, and report latency + token usage."""
    started = time.time()
    response = graph.invoke(state, config=config)
    latency = time.time() - started

    state[channel][-1].pretty_print()
    ai_msg = response[channel][-1]
    ai_msg.pretty_print()

    usage = getattr(ai_msg, "usage_metadata", None)
    if usage:
        tokens = usage.get("total_tokens", 0)
    else:
        meta = getattr(ai_msg, "response_metadata", {}) or {}
        token_usage = meta.get("token_usage") or meta.get("usage") or {}
        tokens = token_usage.get("total_tokens", 0)

    # ANSI: cyan for latency, green for tokens
    print(f"\n\033[96m⏱️  Latency: {latency:.2f}s\033[0m")
    print(f"\033[92m📋 Total tokens: {tokens}\033[0m")

    return response


print("✅ Instrumentation ready!")

---
## 📜 Part 1: Sequential Memory (The Baseline)

The simplest possible policy: **keep everything**. Every turn is appended to state, and the
whole list is sent to the model on every call.

Note the reducer here is `operator.add` — plain list concatenation. It appends and never
removes, which is exactly the behaviour we want to measure before improving on it.

In [ ]:
# ============================================================================
# STATE SCHEMA: Sequential (Append-Only)
# ============================================================================
# `add` is plain list concatenation - nothing is ever dropped.

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add]


print("✅ Sequential state schema defined!")

In [ ]:
# ============================================================================
# CHAT NODE: Send the Entire History
# ============================================================================

def chat_llm_node(state: MessagesState):
    """Prepend the system prompt, then send every message we have."""
    history = [SystemMessage(content="You are a customer support assistant.")]
    history.extend(state["messages"])

    reply = llm.invoke(history)

    return {"messages": [reply]}


print("✅ Sequential chat node defined!")

In [ ]:
# ============================================================================
# GRAPH: Sequential Memory
# ============================================================================
# flow: START -> chat_llm -> END

checkpointer = InMemorySaver()

builder = StateGraph(MessagesState)
builder.add_node("chat_llm", chat_llm_node)
builder.add_edge(START, "chat_llm")
builder.add_edge("chat_llm", END)

graph = builder.compile(checkpointer=checkpointer)

print("✅ Sequential graph compiled!")

### 1.1 🧪 Three Turns, Measured

Watch the token count on each turn. The conversation is short enough that the answers stay
good — the point is the *trend*, not the quality.

In [ ]:
# ============================================================================
# SEQUENTIAL: Turn 1
# ============================================================================

config = {"configurable": {"thread_id": "ticket-seq"}}

response1 = timed_invoke(
    graph,
    {"messages": [HumanMessage(content=(
        "Hi, I'm being charged twice for my subscription. "
        "Can you help me figure out what's going on?"
    ))]},
    config,
)

In [ ]:
# ============================================================================
# SEQUENTIAL: Turn 2
# ============================================================================

response2 = timed_invoke(
    graph,
    {"messages": [HumanMessage(content=(
        "I think this started after I changed "
        "my billing address last month."
    ))]},
    config,
)

In [ ]:
# ============================================================================
# SEQUENTIAL: Turn 3 - Requires Recall
# ============================================================================

response3 = timed_invoke(
    graph,
    {"messages": [HumanMessage(content=(
        "Can you summarize what we did just now?"
    ))]},
    config,
)

### 1.2 📊 What the Numbers Show

**Prompt tokens grow roughly linearly with the number of turns.** You pay for the entire
past conversation on every single call, and response time climbs with prompt size.

For short conversations this is fine — no detail is ever lost, and for internal tools or
early prototypes that may be all you need. But it does not scale:

- **Every extra token has a cost.** An unmanaged history becomes financially unsustainable.
- **Large prompts mean high latency.** Slow answers are unacceptable in production.

Sequential memory is best treated as a **baseline**: the most straightforward behaviour,
easy to instrument, and the thing every other strategy is measured against.

The next step keeps the spirit of it — don't lose what matters — while putting a hard
ceiling on context.

---
## 🪟 Part 2: Sliding Window Memory

Instead of retaining everything, keep only the **most recent N messages**. As new messages
arrive, the oldest drop off and the window slides forward.

The flow becomes:

1. The user and agent take turns; each turn is appended to history
2. Before generating a reply, keep only the last N messages
3. That trimmed window plus the new query goes to the LLM
4. Everything outside the window is forgotten *for this turn*

Think of it as a short-term memory buffer: the agent remembers what just happened, but
starts forgetting the earliest parts of the conversation.

### Key Concepts:
- **`add_messages` reducer**: unlike `add`, it understands message *ids* — which is what
  makes deletion possible
- **`RemoveMessage(id=...)`**: returning one tells the reducer to delete that message from state

In [ ]:
# ============================================================================
# STATE SCHEMA: Sliding Window
# ============================================================================
# Note the reducer change: add_messages (not add). Only add_messages knows how
# to interpret RemoveMessage, which is how we drop old turns.

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


print("✅ Sliding-window state schema defined!")

In [ ]:
# ============================================================================
# TRUNCATION NODE: Enforce the Window
# ============================================================================

MAX_MESSAGES = 4


def truncate_messages_node(state: MessagesState):
    """Drop everything older than the last MAX_MESSAGES messages."""
    msgs = state["messages"]
    print(f"🪟 [truncate] incoming length = {len(msgs)}")

    if len(msgs) <= MAX_MESSAGES:
        print("🪟 [truncate] no-op (under limit)")
        return {}

    to_drop = msgs[:-MAX_MESSAGES]
    print(f"🗑️  [truncate] dropping {len(to_drop)} message(s)")

    # RemoveMessage is a deletion instruction, not a message the model will see.
    return {"messages": [RemoveMessage(id=m.id) for m in to_drop]}


print("✅ Truncation node defined!")

In [ ]:
# ============================================================================
# CHAT NODE: Instrumented to Show What Survived the Window
# ============================================================================

def chat_llm_node(state: MessagesState):
    """Same as before, but prints what the model actually receives."""
    history = [SystemMessage(content="You are a customer support assistant.")]

    print(f"🤖 [chat_llm] LLM sees {len(state['messages'])} message(s):")
    for m in state["messages"]:
        m.pretty_print()

    history.extend(state["messages"])

    reply = llm.invoke(history)

    return {"messages": [reply]}


print("✅ Instrumented chat node defined!")

In [ ]:
# ============================================================================
# GRAPH: Sliding Window Memory
# ============================================================================
# flow: START -> truncate -> chat_llm -> END
# Truncation runs BEFORE the model call, so the window is enforced every turn.

checkpointer = InMemorySaver()

builder = StateGraph(MessagesState)
builder.add_node("truncate", truncate_messages_node)
builder.add_node("chat_llm", chat_llm_node)

builder.add_edge(START, "truncate")
builder.add_edge("truncate", "chat_llm")
builder.add_edge("chat_llm", END)

sliding_graph = builder.compile(checkpointer=checkpointer)

print("✅ Sliding-window graph compiled!")

### 2.1 🧪 The Same Conversation, Windowed

Turn 1 slips in an extra detail — *"My favorite color is blue by the way."* By turn 3 that
message has scrolled out of the window, and the agent genuinely cannot answer.

That failure is the lesson, not a bug.

In [ ]:
# ============================================================================
# SLIDING WINDOW: Turn 1 - Plants a Detail We Will Ask About Later
# ============================================================================

config = {"configurable": {"thread_id": "ticket-sliding"}}

response1 = timed_invoke(
    sliding_graph,
    {"messages": [HumanMessage(content=(
        "Hi, I'm being charged twice for my subscription. "
        "Can you help me figure out what's going on? "
        "My favorite color is blue by the way."
    ))]},
    config,
)

In [ ]:
# ============================================================================
# SLIDING WINDOW: Turn 2
# ============================================================================

response2 = timed_invoke(
    sliding_graph,
    {"messages": [HumanMessage(content=(
        "I think this started after I changed "
        "my billing address last month."
    ))]},
    config,
)

In [ ]:
# ============================================================================
# SLIDING WINDOW: Turn 3 - The Detail Is Gone
# ============================================================================
# Watch the [truncate] output: the message mentioning blue has been dropped,
# so the agent has no way to answer this.

response3 = timed_invoke(
    sliding_graph,
    {"messages": [HumanMessage(content=(
        "What is my favorite color?"
    ))]},
    config,
)

### 2.2 📊 What the Numbers Show

**Tokens per turn grow only until the window fills, then flatten.** Latency follows the same
curve — it rises early, then plateaus once prompt size stabilises. That is the whole win.

The cost is equally clear: **anything outside the window is gone.** For a support agent that
bites when:

- A critical fact was mentioned early (*"I'm on the Enterprise plan"*, *"only happens on Firefox"*)
  and has since scrolled away
- The user refers back to something many turns ago
- Tickets run long and interleave several threads of discussion

Sliding window keeps cost and latency under control, and works well for short, focused
conversations where what matters is recent. It has no way to remember older details.

That is what summarization fixes.

---
## 🧬 Part 3: Summarization-Based Memory

If you have used an AI coding assistant, you have already seen this. As the context window
fills, instead of failing or blindly truncating, the tool quietly:

- compresses older parts of the conversation into short summaries
- retains key decisions, constraints, and naming conventions
- keeps only the most recent turns in full detail

The assistant still remembers what mattered, while the prompt stays a manageable size.

Instead of dropping old information, we **remember it in condensed form**: periodically
summarise the conversation so far and use that summary in place of the full history.

### Key Concepts:
- **Two channels**: `summary` (a plain string, overwritten each time) and `buffer` (recent messages)
- **The cost**: one extra LLM call per summarisation — you trade tokens for tokens

In [ ]:
# ============================================================================
# STATE SCHEMA: Summary + Recent Buffer
# ============================================================================
# `summary` has no reducer, so writing it replaces the old value.
# `buffer` uses add_messages, so recent turns accumulate.

class SummarizationState(TypedDict):
    summary: str
    buffer: Annotated[list[AnyMessage], add_messages]


print("✅ Summarization state schema defined!")

In [ ]:
# ============================================================================
# SUMMARIZATION NODE: Compress the Buffer
# ============================================================================

MESSAGE_THRESHOLD = 4


def summarize_buffer_node(state: SummarizationState):
    """Fold the current buffer into a running summary once it gets long enough."""
    buffer = state["buffer"]
    current_summary = state.get("summary", "")

    if len(buffer) < MESSAGE_THRESHOLD:
        return {}

    # --- Render the buffer as a transcript for the summarizer ---
    lines = []
    for i, m in enumerate(buffer):
        role = "User" if i % 2 == 0 else "Assistant"
        lines.append(f"{role}: {m.content}")
    buffer_text = "\n".join(lines)

    system = SystemMessage(content=(
        "Maintain a summary of a support conversation.\n"
        "Keep only important facts and decisions.\n"
        "Prefer short bullet points when possible."
    ))

    user_content = (
        f"Previous summary:\n{current_summary or '(none)'}\n\n"
        f"New conversation turns:\n{buffer_text}\n\n"
        "Produce an updated summary."
    )

    summary_reply = llm.invoke([system, HumanMessage(content=user_content)])
    new_summary = summary_reply.content

    print(f"🧬 [summarize] compressed {len(buffer)} message(s) into {len(new_summary)} chars")

    return {"summary": new_summary}


print("✅ Summarization node defined!")

In [ ]:
# ============================================================================
# CHAT NODE: Summary for Global Context, Buffer for Local Detail
# ============================================================================

def chat_llm_with_summary_node(state: SummarizationState):
    """Send the running summary and the recent buffer, not the full history."""
    messages = [SystemMessage(content=(
        "You are a helpful customer support assistant. "
        "Use the conversation summary for global context "
        "and the recent messages for local detail."
    ))]

    summary = state.get("summary", "")
    if summary:
        messages.append(SystemMessage(
            content=f"Conversation summary so far:\n{summary}"
        ))

    messages.extend(state["buffer"])

    reply = llm.invoke(messages)

    return {"buffer": [reply]}


print("✅ Summary-aware chat node defined!")

In [ ]:
# ============================================================================
# GRAPH: Summarization Memory
# ============================================================================
# flow: START -> summarize -> chat_llm -> END

checkpointer = InMemorySaver()

builder = StateGraph(SummarizationState)
builder.add_node("summarize", summarize_buffer_node)
builder.add_node("chat_llm", chat_llm_with_summary_node)

builder.add_edge(START, "summarize")
builder.add_edge("summarize", "chat_llm")
builder.add_edge("chat_llm", END)

summary_graph = builder.compile(checkpointer=checkpointer)

print("✅ Summarization graph compiled!")

### 3.1 🧪 Facts Mentioned Early, Recalled Late

Turn 1 describes a problem. Turn 2 supplies the plan and workspace name. Turn 3 asks the
agent to recall both — which only works because they survived in the summary.

> **Note**: Only turn 1 seeds `summary`. Later turns pass **`buffer` only**. Because
> `summary` has no reducer, sending it again would overwrite the accumulated summary with
> whatever you passed — which is exactly how this demo silently fails.

In [ ]:
# ============================================================================
# SUMMARIZATION: Turn 1 - Seeds the Thread
# ============================================================================

config = {"configurable": {"thread_id": "ticket-summarize"}}

response1 = timed_invoke(
    summary_graph,
    {"summary": "", "buffer": [HumanMessage(content=(
        "Hi, my Team Analytics workspace isn't updating our "
        "weekly reports. The data is stuck on last Monday."
    ))]},
    config,
    channel="buffer",
)

In [ ]:
# ============================================================================
# SUMMARIZATION: Turn 2 - Supplies Details Worth Remembering
# ============================================================================
# Only `buffer` is passed - see the note above.

response2 = timed_invoke(
    summary_graph,
    {"buffer": [HumanMessage(content=(
        "We use the Growth plan with 15 seats. "
        "The workspace name is 'Marketing Performance Q4'."
    ))]},
    config,
    channel="buffer",
)

In [ ]:
# ============================================================================
# SUMMARIZATION: Turn 3 - Recall Across the Compression Boundary
# ============================================================================

response3 = timed_invoke(
    summary_graph,
    {"buffer": [HumanMessage(content=(
        "Can you remind me our workspace name "
        "and the problem we are facing?"
    ))]},
    config,
    channel="buffer",
)

In [ ]:
# ============================================================================
# INSPECT: What the Summary Actually Holds
# ============================================================================

final_state = summary_graph.get_state(config)

print("🧬 Running summary:\n")
print(final_state.values.get("summary") or "(no summary yet)")
print(f"\n📋 Messages still in buffer: {len(final_state.values.get('buffer', []))}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Three Strategies, One Trade-Off

| Strategy | Tokens per turn | Remembers old details? | Extra cost |
|---|---|---|---|
| **Sequential** | grows linearly, forever | ✅ everything | none |
| **Sliding window** | flat once window fills | ❌ anything outside the window | none |
| **Summarization** | flat-ish, summary grows slowly | ✅ the gist, not the wording | one extra LLM call |

### 2. The Reducer Is the Mechanism
- **`operator.add`** concatenates — it can only ever grow
- **`add_messages`** tracks message ids, which is what makes **`RemoveMessage`** work
- You cannot implement a sliding window without switching reducers first

### 3. Watch Out for Channels Without Reducers
- **`summary` is a plain `str`**, so any value you pass **replaces** it
- Passing `{"summary": ""}` on every turn silently wipes the memory you are trying to build —
  seed it once, then send only the buffer

### 4. Measure, Don't Assume
- Latency and token usage per turn are the only way to see these curves diverge
- Sliding window flattens both; summarization trades a second model call for a shorter prompt
- Which one is right depends on whether your conversations are *long* or merely *ongoing*

### Next Steps
- **`../02_Long_Term_Memory/`** — memory that persists across threads, not just within one
- **`../../../Deep_Agents_and_Harness_Engineering/`** — context compaction at agent-harness scale
- LangChain 1.x ships **`SummarizationMiddleware`**, which applies Part 3's idea automatically
  to any `create_agent()` agent